# 03 · Events, attention indices and placebo dates

Inputs: `holder_snapshot` (02; six main events, `bip360_draft`, `latest`)

Outputs: `events`, `event_windows`, `attention_daily`, `event_attention`, `placebo_days`, `placebo_draws`, `wiki_frozen`, `gtrends_frozen`

| Step | Table | Content |
|---|---|---|
| 1 | `events` | main events, secondary events, positive controls and confounds, with date precision and verification status |
| 2 | `event_windows` | event × relative day (−60…+60), main window (−14…+30) and overlaps with other main events |
| 3–4 | `attention_daily` | daily Wikipedia pageviews, GDELT news and Google Trends; anomalies; PC1, equal-weight and maximum indices |
| 5 | `event_attention` | attention intensity in each event window |
| 6 | `placebo_days`, `placebo_draws` | Tuesday placebo pool and 1,000 draws for randomization inference; non-quantum news days (negative controls) |

Events are dated from primary sources. The attention indices validate attention, measure intensity and screen placebo dates; they are not used to select events.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

ATTN_START   = "2022-01-01"   # covers the positive controls FTX (2022-11) and Randstorm (2023-11)
LATEST_SNAP  = "2026-09-22"   # same as 02 (freeze date)
ATTN_END     = LATEST_SNAP

BASE_WIN     = 28             # anomaly baseline window (days)
MAIN_WIN     = (-14, 30)      # main event window
EXCL_DAYS    = 30             # robustness: symmetric exclusion of +/-30 days (conservative placebo pool)
# main exclusion: window [d-14, d+30] of placebo date d contains no shock s, i.e. d not in [s-30, s+14]
PRE_EXCL, POST_EXCL = MAIN_WIN[1], -MAIN_WIN[0]
GTR_ZERO_MAX = 0.2            # Google Trends: censored if > 20% of the previous 28 days are 0
NQ_BTC_MIN   = 1.5            # non-quantum news days: threshold on general Bitcoin attention (fixed)
FREEZE_REFRESH = False        # True = re-pull Wikipedia and Google Trends and overwrite the frozen tables
PLACEBO_FROM = "2024-06-04"   # placebo pool start; Tuesdays (modal weekday of the main events)
N_DRAWS, SEED = 1000, 20260923
SPIKE_SD     = 2.0            # max attn_pc1 within 3 days of a placebo date
UA = {"User-Agent": "qbtc-replication/1.0 (contact: your-email@example.org)"}  # Wikimedia asks for a User-Agent with contact details

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pytrends"], check=False)

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, time, datetime as dt, urllib.parse
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt

PLACEBO_TO = (pd.Timestamp(LATEST_SNAP) - pd.Timedelta(days=MAIN_WIN[1])).strftime("%Y-%m-%d")

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

client.get_table(f"{DS}.holder_snapshot")   # raises if the table is missing
print("Tables from 02 found. Spent so far at list price: $%.2f / $%.0f" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "MB": round(t.num_bytes / 2**20, 2)}

def build(name, sql, force=False):
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}; set CONFIRM=True and re-run")
        return
    print(f"[{name}] estimated {gb:,.1f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 2),
                           "usd": round(usd(billed), 3), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 3)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.2f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))

def q(sql):
    return client.query(sql).to_dataframe()

def upload(name, df):
    # DataFrame -> BigQuery (overwrite); date columns stored as DATE
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    job = client.load_table_from_dataframe(df, f"{DS}.{name}", job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"[{name}] uploaded", table_info(name))
    df.to_csv(f"{name}.csv", index=False)

# frozen external data: read from BigQuery after the first pull
def load_frozen(name):
    if FREEZE_REFRESH:
        return None
    try:
        df = client.query(f"SELECT * FROM `{DS}.{name}` ORDER BY date").to_dataframe()
    except Exception:
        print(f"[{name}] no frozen table yet; pulling and freezing")
        return None
    df["date"] = pd.to_datetime(df.date)
    df = df.set_index("date")
    if df.index.min() > pd.Timestamp(ATTN_START) or df.index.max() < pd.Timestamp(ATTN_END):
        print(f"[{name}] frozen table covers only {df.index.min().date()}-{df.index.max().date()}, not {ATTN_START}-{ATTN_END}; pulling again")
        return None
    print(f"[{name}] reading the frozen version (pulled at {df.pulled_at.iloc[0] if 'pulled_at' in df else '?'})")
    return df.drop(columns=[c for c in ["pulled_at"] if c in df]).apply(pd.to_numeric, errors="coerce").astype(float).reindex(days)

def save_frozen(name, df):
    out = df.copy()
    out.index.name = "date"
    out = out.reset_index()
    out["pulled_at"] = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    upload(name, out)            # also writes {name}.csv

## Step 1 · `events`

All dates verified against primary sources on 24 September 2026, day precision. `date_status`: `primary_verified` (paper, repository or official announcement), `author_announcement`, `contemporaneous_report`, `chain_verified` (block height). `in_analysis` is True for main events, secondary events and positive controls; confounds only flag and exclude placebo dates. `alt_date` is an earlier public milestone, used to re-estimate with the event moved earlier.

In [ ]:
# Step 1: events (dates verified 2026-09-24)
# Six main events; bip360/bip360_draft and lopp/bip361 are two stages of the same pull request
E = [
 # event, date, event_set, kind, channel, date_status, source / note
 ("willow",        "2024-12-09", "main", "hardware",   "risk_belief", "primary_verified",
  "Google officially announces Willow; window includes 2024-12-17 Bitcoin-specific media wave (Fortune on migration downtime, Bitcoin Frontier podcast)"),
 ("gidney",        "2025-05-21", "main", "science",    "risk_belief", "primary_verified",
  "Gidney RSA-2048 <1m noisy qubits; arXiv v1"),
 ("lopp",          "2025-07-15", "main", "governance", "penalty",     "primary_verified",
  "BIP-361 initial PR #1895 / legacy-signature sunset proposal"),
 ("bip360",        "2026-02-11", "main", "governance", "wait_option", "primary_verified",
  "BIP-360 P2MR (PR #1670) merged into bitcoin/bips; same day BIP number 361 assigned to PR #1895"),
 ("google",        "2026-03-31", "main", "science",    "risk_belief", "primary_verified",
  "Google Research public disclosure of ECDLP-256 resource estimates; arXiv v1 was 2026-03-30"),
 ("bip361",        "2026-04-14", "main", "governance", "penalty",     "primary_verified",
  "BIP-361 (PR #1895) merged into bitcoin/bips"),

 ("bip360_draft",  "2024-09-27", "secondary", "governance",  "wait_option", "primary_verified",
  "BIP-360 initial upstream PR #1670; BIP number 360 assigned 2024-12-18 (inside Willow window)"),
 ("hourglass",     "2025-04-29", "secondary", "governance",  "penalty",     "primary_verified",
  "Hourglass proposal posted to Bitcoin Development mailing list"),
 ("blackrock",     "2025-05-09", "secondary", "institution", "risk_belief", "primary_verified",
  "IBIT SEC filing materially expands quantum-computing risk disclosure"),
 ("chaincode",     "2025-05-28", "secondary", "institution", "risk_belief", "author_announcement",
  "Authors publicly announce Chaincode Bitcoin and Quantum Computing report"),
 ("citi",          "2026-01-15", "secondary", "institution", "risk_belief", "primary_verified",
  "Citi Institute Quantum Threat report"),
 ("qsb",           "2026-04-09", "secondary", "tool",        "cost",        "primary_verified",
  "StarkWare Quantum-Safe Bitcoin Transactions Without Softforks"),
 ("pacts",         "2026-05-01", "secondary", "governance",  "penalty",     "primary_verified",
  "Paradigm PACTs proposal"),
 ("glassnode",     "2026-05-20", "secondary", "institution", "risk_belief", "primary_verified",
  "Glassnode Measuring Bitcoin's Quantum-Exposed Supply"),
 ("starkware_tx",  "2026-08-26", "secondary", "tool",        "cost",        "primary_verified",
  "StarkWare reports first QSB transaction mined on Bitcoin mainnet; window overlaps BIP-110 split and eCash Alpha"),

 ("ftx",           "2022-11-08", "positive_control", "exchange", None, "contemporaneous_report",
  "FTX.com withdrawals effectively paused; bankruptcy 2022-11-11"),
 ("randstorm",     "2023-11-14", "positive_control", "wallet",   None, "primary_verified",
  "Randstorm full technical disclosure; vulnerability pre-announced 2023-10-10"),

 ("spot_etf",      "2024-01-10", "confound", "market",     None, "primary_verified",
  "SEC approves US spot Bitcoin ETPs"),
 ("halving",       "2024-04-20", "confound", "protocol",   None, "chain_verified",
  "Fourth Bitcoin halving at block 840000"),
 ("bip110_signal", "2026-08-08", "confound", "governance", None, "chain_verified",
  "BIP-110 mandatory signalling from block 961632 (2026-08-08 19:35:55 UTC); not activated on main chain, BIP-110 nodes split onto a minority branch"),
 ("ecash_alpha",   "2026-08-23", "confound", "fork",       None, "chain_verified",
  "eCash Alpha fork/snapshot at ~block 963648 (practice stage); final mainnet hard fork scheduled 2026-10-31 (~block 973728), after freeze"),

 ("world_quantum_day_2026", "2026-04-14", "confound", "commemorative", None, "primary_verified",
  "World Quantum Day (annual, 14 April); 2026: Wikipedia 'Quantum computing' human pageviews ~14x (desktop+mobile), bitcoin-specific components flat; same day as BIP-361 merge"),

 # attention drivers (media events): placebo exclusion only
 ("china_dwave_media", "2024-10-11", "attention_driver", "media", "risk_belief", "primary_verified",
  "SCMP report triggers media wave over Shanghai University D-Wave cryptanalysis paper; underlying paper online 2024-09-30 (CNKI)"),
]
events = pd.DataFrame(E, columns=["event", "event_date", "event_set", "kind", "channel", "date_status", "note"])
events["event_date"] = pd.to_datetime(events.event_date)

# earlier public milestones (alt_date robustness)
ALT = {
    "google":    ("2026-03-30", "arXiv v1 one day before public disclosure; t-1 snapshot falls on it"),
    "randstorm": ("2023-10-10", "vulnerability pre-announced"),
    "ftx":       ("2022-11-11", "bankruptcy filing"),
}
events["alt_date"] = pd.to_datetime(events.event.map(lambda e: ALT.get(e, (None,))[0]))
events["alt_note"] = events.event.map(lambda e: ALT.get(e, (None, None))[1])
events["block_height"] = events.event.map({"halving": 840000, "bip110_signal": 961632, "ecash_alpha": 963648}).astype("Int64")

VERIFIED = ["primary_verified", "author_announcement", "contemporaneous_report", "chain_verified"]
assert events.date_status.isin(VERIFIED).all(), events.loc[~events.date_status.isin(VERIFIED), ["event", "date_status"]]
events["date_precision"] = "day"
events["in_analysis"]    = events.event_set.isin(["main", "secondary", "positive_control"])   # confound / attention_driver: flagging and exclusion only
# starkware_tx: window contains the BIP-110 fork and eCash Alpha; bip360_draft: [0,+30] contains the 2024-10-11 D-Wave media storm
events["confounded"]     = events.event.isin(["starkware_tx", "bip360_draft"])
# attention confounded: [0,+7] of bip361 and qsb contains World Quantum Day (2026-04-14)
events["attn_confounded"] = events.event.isin(["bip361", "qsb"])
events["snap_date"]      = events.event_date - pd.Timedelta(days=1)

# reconcile with the events in the 02 snapshots
snap_ev = q(f"SELECT DISTINCT event, event_date FROM `{DS}.holder_snapshot` WHERE event != 'latest'")
chk = events.merge(snap_ev.assign(event_date=pd.to_datetime(snap_ev.event_date)),
                   on="event", how="right", suffixes=("", "_02"))
bad = chk[chk.event_date != chk.event_date_02]
assert bad.empty, f"events and 02 snapshot dates disagree:\n{bad}"
print("Consistent with the 02 snapshot events:", sorted(snap_ev.event))

# pairwise overlap of main events (window -14...+30)
m = events[events.event_set == "main"].sort_values("event_date")
ov = []
for _, a in m.iterrows():
    for _, b in m.iterrows():
        if a.event < b.event:
            lo = max(a.event_date, b.event_date) + pd.Timedelta(days=MAIN_WIN[0])
            hi = min(a.event_date, b.event_date) + pd.Timedelta(days=MAIN_WIN[1])
            if lo <= hi:
                ov.append((a.event, b.event, abs((b.event_date - a.event_date).days), (hi - lo).days + 1))
print("Overlapping main-event windows (joint estimation):", ov)

# secondary events inside main-event windows (care needed when estimated separately)
for _, s in events[events.event_set == "secondary"].iterrows():
    hit = [r.event for _, r in m.iterrows()
           if MAIN_WIN[0] <= (s.event_date - r.event_date).days <= MAIN_WIN[1]]
    if hit: print(f"  secondary event {s.event} ({s.event_date:%Y-%m-%d}) is inside main-event windows: {hit}")

display(events[["event", "event_date", "event_set", "channel", "date_status", "alt_date", "in_analysis", "confounded"]])
upload("events", events)

# attention-spike audit (identified by hand, 2026-09-24); not treatment events
# driver_type: technical = substantive technical/standards news; media = media wave; bitcoin_specific = directly about Bitcoin's quantum risk
SPIKE_AUDIT = [
 # spike_date,  source_date,  label,                                   driver_type,        confidence, url
 ("2022-01-27", "2022-01-27", "BBC: What is the quantum apocalypse (mentions Bitcoin wallets drained)", "media", "high",
  "https://crucialcompass.squarespace.com/data"),
 ("2022-07-06", "2022-07-05", "NIST announces first four PQC algorithms",  "technical", "high",
  "https://www.nist.gov/news-events/news/2022/07/nist-announces-first-four-quantum-resistant-cryptographic-algorithms"),
 ("2022-09-04", "2022-09-04", "Vitalik Buterin post: quantum breaks RSA/elliptic curves, not hashes/lattices (Wikipedia PQC ~4x on a Sunday)", "public_comment", "high",
  "https://decrypt.co/294384/from-the-quantum-realm-to-reality-a-beginner-guide-to-the-computer-of-the-future"),
 ("2023-03-21", "2023-03-21", "NVIDIA DGX Quantum",                         "technical", "medium_high",
  "https://nvidianews.nvidia.com/news/nvidia-announces-new-system-for-accelerated-quantum-classical-computing"),
 ("2023-07-08", "2023-04-21", "Google Sycamore '47 years' media wave (preprint 2023-04-21)", "media", "medium_high",
  "https://arxiv.org/abs/2304.11119"),
 ("2023-12-04", "2023-12-04", "IBM Quantum Summit: Heron, Quantum System Two", "technical", "high",
  "https://newsroom.ibm.com/2023-12-04-IBM-Debuts-Next-Generation-Quantum-Processor-IBM-Quantum-System-Two%2C-Extends-Roadmap-to-Advance-Era-of-Quantum-Utility"),
 ("2024-07-28", "2024-07-28", "CBS 60 Minutes quantum computing feature",  "media", "high",
  "https://www.cbsnews.com/news/quantum-computing-google-ibm-advances-60-minutes-transcript/"),
 ("2024-10-14", "2024-10-11", "SCMP on Shanghai Univ. D-Wave cryptanalysis (paper online 2024-09-30)", "media", "high",
  "https://www.scmp.com/news/china/science/article/3282051/chinese-scientists-hack-military-grade-encryption-quantum-computer-paper"),
 ("2024-12-17", "2024-12-17", "Fortune: Bitcoin migration downtime; Bitcoin Frontier podcast (post-Willow)", "bitcoin_specific", "high",
  "https://fortune.com/2024/12/17/bitcoin-downtime-update-quantum-computing-attack-study/"),
 ("2025-02-20", "2025-02-19", "Microsoft Majorana 1",                       "technical", "high",
  "https://blogs.microsoft.com/blog/2025/02/19/microsoft-unveils-majorana-1/"),
 ("2026-04-14", "2026-04-14", "World Quantum Day 2026 (Wikipedia 'Quantum computing' ~14x human views; same day as BIP-361 merge)", "commemorative", "high",
  "https://www.wisdomtree.com/us/insights/blog/quantum-computing-is-entering-the-market-phase-world-quantum-day-2026-showed-it"),
]

## Step 2 · `event_windows`

Day-precision events × relative days −60…+60. `in_main_window` marks the main window; `overlaps_main` lists other main events whose main window contains the same day (used for the joint Google × BIP-361 estimate). Joined in 05.

In [ ]:
SQL_WINDOWS = f'''
CREATE OR REPLACE TABLE `{DS}.event_windows` AS
WITH ew AS (
  SELECT e.event, e.event_set, e.event_date, k AS rel_day,
         DATE_ADD(e.event_date, INTERVAL k DAY) AS date
  FROM `{DS}.events` e, UNNEST(GENERATE_ARRAY(-60, 60)) AS k
  WHERE e.date_precision = 'day'
),
ov AS (   -- other main events whose main window contains the same day (explicit JOIN; BigQuery does not support correlated subqueries on other tables)
  SELECT w.event, w.rel_day, ARRAY_AGG(o.event ORDER BY o.event) AS overlaps_main
  FROM ew w
  JOIN `{DS}.events` o
    ON o.event_set = 'main' AND o.event != w.event
   AND w.date BETWEEN DATE_ADD(o.event_date, INTERVAL {MAIN_WIN[0]} DAY)
                  AND DATE_ADD(o.event_date, INTERVAL {MAIN_WIN[1]} DAY)
  GROUP BY w.event, w.rel_day
)
SELECT
  w.event, w.event_set, w.event_date, w.rel_day, w.date,
  w.rel_day BETWEEN {MAIN_WIN[0]} AND {MAIN_WIN[1]} AS in_main_window,
  IFNULL(ov.overlaps_main, ARRAY<STRING>[]) AS overlaps_main
FROM ew w
LEFT JOIN ov USING (event, rel_day)
'''
build("event_windows", SQL_WINDOWS, force=True)
display(q(f'''
SELECT event, COUNTIF(in_main_window AND ARRAY_LENGTH(overlaps_main) > 0) AS overlap_days,
       STRING_AGG(DISTINCT o, ',') AS with_events
FROM `{DS}.event_windows` LEFT JOIN UNNEST(overlaps_main) o
WHERE event_set = 'main' GROUP BY event ORDER BY event
'''))

## Step 3 · Attention data

| Component | Source | Quantum-Bitcoin attention | General Bitcoin attention (negative controls) |
|---|---|---|---|
| Wikipedia | Wikimedia pageviews API (English, user agents) | Shor's algorithm, Post-quantum cryptography, Quantum computing, Quantum supremacy | Bitcoin |
| GDELT | BigQuery `gdelt-bq.gdeltv2.gkg_partitioned`, URL column only | URLs containing quantum and bitcoin/crypto/blockchain | URLs containing bitcoin |
| Google Trends | pytrends, daily data in 240-day windows chained on 60-day overlaps | "bitcoin quantum" | n/a |

Wikipedia and Google Trends are frozen after the first pull as `wiki_frozen` and `gtrends_frozen` (with pull time) and read from there afterwards, since Google Trends re-samples on every pull. Both tables are in the replication data.

Without a Google Trends series the index uses the other two sources; `gtrends_manual.csv` (columns date,value) is read first if present.

In [ ]:
# 3a - Wikipedia pageviews
WIKI = {
    "wiki_shor":    "Shor's algorithm",
    "wiki_pqc":     "Post-quantum cryptography",
    "wiki_qc":      "Quantum computing",
    "wiki_qsup":    "Quantum supremacy",
    "wiki_bitcoin": "Bitcoin",
}
def wiki_daily(title, start, end, project="en.wikipedia"):
    a = urllib.parse.quote(title.replace(" ", "_"), safe="")
    url = (f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/{project}/all-access/user/"
           f"{a}/daily/{pd.Timestamp(start):%Y%m%d}00/{pd.Timestamp(end):%Y%m%d}00")
    for i in range(4):
        r = requests.get(url, headers=UA, timeout=60)
        if r.status_code == 200: break
        time.sleep(3 * (i + 1))
    r.raise_for_status()
    it = r.json()["items"]
    return pd.Series({pd.Timestamp(x["timestamp"][:8]): x["views"] for x in it}, dtype=float)

days = pd.date_range(ATTN_START, ATTN_END, freq="D")
wiki = load_frozen("wiki_frozen")
if wiki is None:
    wiki = pd.DataFrame(index=days)
    for col, title in WIKI.items():
        s = wiki_daily(title, ATTN_START, ATTN_END)
        wiki[col] = s.reindex(days).fillna(0)          # the API does not return days with 0 views
        print(f"{col:13s} {title:28s} days {s.size:5d}  median {s.median():,.0f}  max {s.max():,.0f} ({s.idxmax():%Y-%m-%d})")
    save_frozen("wiki_frozen", wiki)
wiki.index.name = "date"

In [ ]:
# 3b - GDELT (DocumentIdentifier column only)
SQL_GDELT = f'''
CREATE OR REPLACE TABLE `{DS}.gdelt_daily` AS
SELECT
  DATE(pt) AS date,
  COUNT(*) AS n_all,
  COUNTIF(REGEXP_CONTAINS(u, r'quantum') AND REGEXP_CONTAINS(u, r'bitcoin|crypto|blockchain|satoshi')) AS n_qbtc,
  COUNTIF(REGEXP_CONTAINS(u, r'quantum'))  AS n_quantum,
  COUNTIF(REGEXP_CONTAINS(u, r'bitcoin'))  AS n_bitcoin,
  COUNTIF(REGEXP_CONTAINS(u, r'bitcoin|crypto|blockchain|satoshi')) AS n_crypto
FROM (
  SELECT _PARTITIONTIME AS pt, LOWER(DocumentIdentifier) AS u
  FROM `gdelt-bq.gdeltv2.gkg_partitioned`
  WHERE _PARTITIONTIME >= TIMESTAMP('{ATTN_START}')
    AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE '{ATTN_END}', INTERVAL 1 DAY))
)
GROUP BY date
'''
# rebuild if n_crypto is missing or the table does not cover ATTN_START-ATTN_END
try:
    _cols = [f.name for f in client.get_table(f"{DS}.gdelt_daily").schema]
    _rng = q(f"SELECT MIN(date) AS lo, MAX(date) AS hi FROM `{DS}.gdelt_daily`").iloc[0]
    _covered = (pd.Timestamp(_rng.lo) <= pd.Timestamp(ATTN_START)) and (pd.Timestamp(_rng.hi) >= pd.Timestamp(ATTN_END))
except Exception:
    _cols, _covered = [], False
build("gdelt_daily", SQL_GDELT, force=("n_crypto" not in _cols) or not _covered)
gd = q(f"SELECT * FROM `{DS}.gdelt_daily` ORDER BY date")
gd["date"] = pd.to_datetime(gd.date)
gd = gd.set_index("date").reindex(days)
gd = gd.apply(pd.to_numeric, errors="coerce").astype(float)   # BigQuery integer columns are Int64; convert to float to allow NaN
missing = gd.n_all.isna() | (gd.n_all < 0.2 * gd.n_all.rolling(28, min_periods=7).median())
print(f"GDELT covers {gd.n_all.notna().sum()} / {len(days)} days; {missing.sum()} days with gaps or very few articles (set to missing)")
gd.loc[missing, :] = np.nan
# Bitcoin-specific: quantum share of crypto articles (per thousand); missing below 20 crypto articles
gdelt = pd.DataFrame({
    "gdelt_qbtc":    (gd.n_qbtc / gd.n_crypto * 1e3).where(gd.n_crypto >= 20),
    "gdelt_bitcoin": gd.n_bitcoin / gd.n_all * 1e6,
}, index=days)
print(gdelt.describe().round(1))

In [ ]:
# 3c - Google Trends (skipped if the pull fails)
GT_KW, GT_WIN, GT_OVERLAP = "bitcoin quantum", 240, 60

def gtrends_daily(kw, start, end):
    from pytrends.request import TrendReq
    pt = TrendReq(hl="en-US", tz=0)   # no retries/backoff arguments: pytrends is incompatible with recent urllib3 (method_whitelist); retry manually
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    parts, s = [], start
    while True:
        e = min(s + pd.Timedelta(days=GT_WIN - 1), end)
        for tr in range(4):
            try:
                pt.build_payload([kw], timeframe=f"{s:%Y-%m-%d} {e:%Y-%m-%d}", geo="")
                d = pt.interest_over_time()
                break
            except Exception as ex:
                if tr == 3: raise
                print(f"  retry {tr + 1}: {repr(ex)[:80]}"); time.sleep(30 * (tr + 1))
        parts.append(d[kw].astype(float))
        print(f"  {s:%Y-%m-%d} -> {e:%Y-%m-%d}  {len(d)} days")
        if e >= end: break
        s = e - pd.Timedelta(days=GT_OVERLAP - 1)
        time.sleep(10)
    out = parts[0]
    for p in parts[1:]:                     # rescale each window to the previous one with the ratio of means on the overlap
        ov_ = out.index.intersection(p.index)
        a, b = out.loc[ov_].mean(), p.loc[ov_].mean()
        k = a / b if b > 0 and a > 0 else np.nan
        if not np.isfinite(k):
            print("  overlap is 0; this window is joined 1:1"); k = 1.0
        out = pd.concat([out, p.loc[p.index > out.index.max()] * k])
    return 100 * out / out.max()

# Google Trends re-samples on every pull: freeze after the first pull
gtr = load_frozen("gtrends_frozen")
if gtr is None:
    gtr = pd.DataFrame(index=days)
    if os.path.exists("gtrends_manual.csv"):
        m_ = pd.read_csv("gtrends_manual.csv", parse_dates=["date"]).set_index("date")["value"]
        gtr["gtr_qbtc"] = m_.reindex(days)
        print("Using the manually downloaded gtrends_manual.csv")
    else:
        try:
            gtr["gtr_qbtc"] = gtrends_daily(GT_KW, ATTN_START, ATTN_END).reindex(days)
        except Exception as ex:
            print("Google Trends pull failed; skipped (not frozen, try again later):", repr(ex)[:200])
    if "gtr_qbtc" in gtr and gtr.gtr_qbtc.notna().any():
        save_frozen("gtrends_frozen", gtr)
gtr.index.name = "date"
print("Google Trends columns:", list(gtr.columns))

## Step 4 · Attention indices

1. Anomaly: log(1+x) minus the median of the same weekday in the previous four weeks (t−7, t−14, t−21, t−28), standardized by the full-sample standard deviation.
2. `attn_pc1` (general quantum attention): first principal component of the four Wikipedia quantum articles and GDELT, fitted on the full sample (a missing component counts as 0 that day); sign set so that loadings sum to a positive number; rescaled to SD 1.
3. Robustness: `attn_ew` (equal-weight average), `attn_max` (maximum component).
4. `attn_btcq` (Bitcoin-specific quantum attention; main measure for event validation): equal-weight average of GDELT (share of crypto articles mentioning quantum) and Google Trends "bitcoin quantum". Google Trends zeros below its privacy threshold are treated as censored: on days when more than 20% of the previous 28 days are 0, the component is missing and `attn_btcq` is GDELT alone.
5. `btc_attn`: general Bitcoin attention (Wikipedia Bitcoin + GDELT bitcoin), used only to select non-quantum news days.
6. `attn_level`: equal-weight z-score of log levels of the quantum components, for the long-run trend plot.

In [ ]:
raw = pd.concat([wiki, gdelt, gtr], axis=1).apply(pd.to_numeric, errors="coerce").astype(float)
raw.index.name = "date"
# censor Google Trends where the zero share over the previous 28 days > GTR_ZERO_MAX
if "gtr_qbtc" in raw:
    _z28 = (raw.gtr_qbtc == 0).rolling(28, min_periods=14).mean().shift(1)
    _ok = _z28 < GTR_ZERO_MAX
    print(f"Google Trends usable days {int(_ok.sum())} / {len(raw)}; first usable day {raw.index[_ok.values.argmax()].date() if _ok.any() else None}")
    print("  share of zeros by year:", (raw.gtr_qbtc == 0).groupby(raw.index.year).mean().round(2).to_dict())
    raw["gtr_qbtc"] = raw.gtr_qbtc.where(_ok)
Q_COLS = [c for c in ["wiki_shor", "wiki_pqc", "wiki_qc", "wiki_qsup", "gdelt_qbtc"] if c in raw and raw[c].notna().sum() > 300]
G_COLS = [c for c in ["gtr_qbtc"] if c in raw and raw[c].notna().sum() > 60]   # Bitcoin-specific index only
B_COLS = [c for c in ["wiki_bitcoin", "gdelt_bitcoin"] if c in raw and raw[c].notna().sum() > 300]
print("Quantum components:", Q_COLS, "\nBitcoin components:", B_COLS)

def abnormal(s, weeks=BASE_WIN // 7):
    # baseline: same-weekday median over the previous 4 weeks (>= 2 non-missing)
    x = np.log1p(s)
    lags = pd.concat([x.shift(7 * k) for k in range(1, weeks + 1)], axis=1)
    base = lags.median(axis=1, skipna=True).where(lags.notna().sum(axis=1) >= 2)
    a = x - base
    return a / a.std()

Z = pd.DataFrame({c: abnormal(raw[c]) for c in Q_COLS + G_COLS + B_COLS})

# PCA (SVD), fitted on the full sample
X = Z[Q_COLS].dropna()
Xc = (X - X.mean()) / X.std()
U, S, Vt = np.linalg.svd(Xc.values, full_matrices=False)
load = pd.Series(Vt[0], index=Q_COLS)
if load.sum() < 0: load = -load
var_share = S[0]**2 / (S**2).sum()
score = ((Z[Q_COLS] - X.mean()) / X.std()).fillna(0) @ load
score[Z[Q_COLS].isna().all(axis=1)] = np.nan
attn_pc1 = (score - score.mean()) / score.std()
print(f"PC1 explains {var_share:.1%} of variance; loadings:", load.round(3).to_dict())

att = pd.DataFrame(index=days)
att.index.name = "date"
att = att.join(raw[Q_COLS + G_COLS + B_COLS]).join(Z.add_prefix("z_"))
att["attn_pc1"]   = attn_pc1
att["attn_ew"]    = Z[Q_COLS].mean(axis=1, skipna=True)
att["attn_max"]   = Z[Q_COLS].max(axis=1, skipna=True)
BTCQ_COLS = [c for c in ["gdelt_qbtc", "gtr_qbtc"] if c in Q_COLS + G_COLS]
att["attn_btcq"]  = Z[BTCQ_COLS].mean(axis=1, skipna=True) if BTCQ_COLS else np.nan
att["btc_attn"]   = Z[B_COLS].mean(axis=1, skipna=True)
lv = np.log1p(raw[Q_COLS])
att["attn_level"] = ((lv - lv.mean()) / lv.std()).mean(axis=1)
print("Bitcoin-specific components:", BTCQ_COLS)
wd = att.attn_pc1.groupby(att.index.dayofweek).mean().round(2)
print("Mean attn_pc1 by weekday (should be close to 0 after de-seasonalizing):", wd.to_dict())
print(att[["attn_pc1", "attn_ew", "attn_max", "attn_btcq", "btc_attn"]].corr().round(2))

upload("attention_daily", att.reset_index())
PCA_INFO = {"components": Q_COLS, "loadings": load.round(4).to_dict(), "var_share": round(float(var_share), 4)}

## Step 5 · Attention in event windows

For each day-precision event: baseline = mean over [−30, −2]; post-event mean and peak over [0, +7]; `pct_rank` = percentile of the 8-day post-event mean among all 8-day rolling windows. Empirical p-values (noise levels differ by year): `p_btcq` = share of 8-day windows in the same calendar year whose `attn_btcq` peak is at least the event's [0, +7] peak (main); `p_pc1` likewise for `attn_pc1`. Reference windows contain no event day and no identified spike.

`peak_shared_with`: other events whose [0, +7] contains the peak day. `base_contaminated_by`: events whose [0, +7] falls in this event's baseline [−30, −2]. These measures are descriptive; events are not added or dropped on this basis.

In [ ]:
roll8 = att.attn_pc1.rolling(8).mean().shift(-7)          # 8-day mean starting on each day
max8 = {c: att[c].rolling(8).max().shift(-7) for c in ["attn_btcq", "attn_pc1"]}   # 8-day peak starting on each day
# reference windows: [t, t+7] with no event day (incl. confounds, attention drivers) and no identified spike
_shock = pd.Series(False, index=att.index)
for d0 in events.event_date:
    _shock.loc[d0: d0] = True
for a0, b0 in [(pd.Timestamp(x[1]), pd.Timestamp(x[0])) for x in SPIKE_AUDIT]:
    _shock.loc[a0: b0] = True
_clean_start = ~_shock.astype(int).rolling(8).max().shift(-7).fillna(1).astype(bool)
def emp_p(col, d, peak):
    ref = max8[col][(max8[col].index.year == d.year) & _clean_start].dropna()
    return float((ref >= peak).mean()) if pd.notna(peak) and len(ref) else np.nan
rows = []
for _, e in events[events.date_precision == "day"].iterrows():
    d = e.event_date
    if d < att.index.min() + pd.Timedelta(days=30) or d > att.index.max(): continue
    w = lambda a, b, col="attn_pc1": att.loc[d + pd.Timedelta(days=a): d + pd.Timedelta(days=b), col]
    post, base = w(0, 7), w(-30, -2)
    rows.append({
        "event": e.event, "event_set": e.event_set, "event_date": d,
        "base_pc1": base.mean(), "post_pc1": post.mean(), "peak_pc1": post.max(),
        "peak_rel_day": (post.idxmax() - d).days if post.notna().any() else None,
        "post_ew": w(0, 7, "attn_ew").mean(), "post_max": w(0, 7, "attn_max").mean(),
        "post_btc": w(0, 7, "btc_attn").mean(),
        "base_btcq": w(-30, -2, "attn_btcq").mean(), "post_btcq": w(0, 7, "attn_btcq").mean(),
        "peak_btcq": w(0, 7, "attn_btcq").max(),
        "peak_date": post.idxmax() if post.notna().any() else pd.NaT,
        "pct_rank": (roll8 < roll8.get(d, np.nan)).mean() if pd.notna(roll8.get(d, np.nan)) else np.nan,
    })
ev_att = pd.DataFrame(rows)
ev_att["jump_pc1"] = ev_att.post_pc1 - ev_att.base_pc1
ev_att["jump_btcq"] = ev_att.post_btcq - ev_att.base_btcq
ev_att["p_btcq"] = [emp_p("attn_btcq", r.event_date, r.peak_btcq) for r in ev_att.itertuples()]   # main measure
ev_att["p_pc1"]  = [emp_p("attn_pc1",  r.event_date, r.peak_pc1)  for r in ev_att.itertuples()]
ev_att = ev_att.merge(events[["event", "attn_confounded"]], on="event", how="left")
ed = events.set_index("event").event_date
ev_att["peak_shared_with"] = [
    ",".join(o for o, od in ed.items() if o != r.event and pd.notna(r.peak_date) and 0 <= (r.peak_date - od).days <= 7)
    for r in ev_att.itertuples()]
ev_att["base_contaminated_by"] = [
    ",".join(o for o, od in ed.items() if o != r.event and -37 <= (od - r.event_date).days <= -2)
    for r in ev_att.itertuples()]
display(ev_att[["event", "event_set", "event_date", "peak_btcq", "p_btcq", "peak_pc1", "p_pc1",
                "attn_confounded", "peak_shared_with", "base_contaminated_by"]].sort_values("event_date").round(3))
upload("event_attention", ev_att)

### PC1 decomposition

Each day's `attn_pc1` split into component contributions (component standardized with the PCA mean and SD, times loading, on the index scale; columns plus constant sum to `attn_pc1`), with raw values.

In [ ]:
_ssd, _smu = score.std(), score.mean()
def pc1_decomp(d0, d1):
    idx = pd.date_range(d0, d1)
    zs = ((Z[Q_COLS] - X.mean()) / X.std()).reindex(idx)
    contrib = zs.fillna(0).mul(load, axis=1) / _ssd
    out = contrib.add_prefix("c_")
    out["const"] = -_smu / _ssd
    out["attn_pc1"] = att.attn_pc1.reindex(idx)
    out["attn_btcq"] = att.attn_btcq.reindex(idx)
    out = out.join(raw[Q_COLS + G_COLS].reindex(idx).add_prefix("raw_"))
    out.index.name = "date"
    return out

DECOMP_WINDOWS = {
    "bip361": ("2026-04-10", "2026-04-16"),
    "google": ("2026-03-28", "2026-04-03"),
    "willow": ("2024-12-06", "2024-12-19"),
    "china_dwave_media": ("2024-10-08", "2024-10-18"),
}
dec = pd.concat({k: pc1_decomp(*v) for k, v in DECOMP_WINDOWS.items()}, names=["window"])
chk_ = (dec.filter(like="c_").sum(axis=1) + dec.const - dec.attn_pc1).abs().max()
print(f"Decomposition check: max difference between the sum of contributions and attn_pc1 {chk_:.2e} (should be ~0)")
with pd.option_context("display.width", 250, "display.max_columns", 40):
    display(dec.loc["bip361"].round(2))
dec.reset_index().to_csv("pc1_decomposition.csv", index=False)

In [ ]:
# Figure: attention index and event days (single series, events marked with vertical lines)
fig, ax = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.4, 1.4]})
ax[0].plot(att.index, att.attn_level.rolling(7).mean(), lw=1.5, color="#2a4d8f")
ax[0].set_ylabel("Level (z, 7-day mean)")
ax[1].plot(att.index, att.attn_pc1, lw=1, color="#2a4d8f")
ax[1].set_ylabel("Abnormal attention, PC1 (sd)")
ax[2].plot(att.index, att.attn_btcq, lw=1, color="#2a4d8f")
ax[2].set_ylabel("Bitcoin-quantum attention (sd)")
for _, e in events[(events.date_precision == "day") & events.event_set.isin(["main", "positive_control"])].iterrows():
    for a in ax:
        a.axvline(e.event_date, color="#888888", lw=0.8, ls="--")
    ax[1].annotate(e.event, (e.event_date, ax[1].get_ylim()[1]), rotation=90, va="top", ha="right",
                   fontsize=8, color="#444444")
for a in ax:
    a.spines[["top", "right"]].set_visible(False); a.grid(axis="y", color="#e5e5e5", lw=0.6)
plt.tight_layout(); plt.savefig("fig_attention.png", dpi=200); plt.show()

## Step 6 · Placebo dates

`ri_pool`: every Tuesday from `PLACEBO_FROM` to `LATEST_SNAP − 30` (the modal weekday of the main events; the snapshot falls on Monday, t−1). A date d is excluded if its window [d−14, d+30] contains a shock, i.e. d ∈ [s−30, s+14] for any event (main, secondary, positive control, confound, attention driver) or spike in `SPIKE_AUDIT`, or if `attn_pc1` ≥ 2 within 3 days. `ri_pool_sym30` uses a symmetric ±30-day exclusion. 1,000 draws of 6 dates per pool (without replacement, fixed seed) are stored in `placebo_draws` (column `pool`) and read by all later inference.

`nonquantum_news` (negative control): same exclusion rules, Monday to Friday; `btc_attn` ≥ 1.5 and `attn_pc1` < 0.5 on every day within ±3 days; chosen greedily by `btc_attn`, at least 30 days apart, at most 6.

`attention_spikes`: days at least 7 days from any event with `attn_pc1` ≥ 3 (at least 14 days apart), listed for manual identification against primary sources; not used as events automatically.

In [ ]:
audit = pd.DataFrame(SPIKE_AUDIT, columns=["spike_date", "source_date", "label", "driver_type", "confidence", "url"])
audit[["spike_date", "source_date"]] = audit[["spike_date", "source_date"]].apply(pd.to_datetime)
audit = audit.merge(att[["attn_pc1", "attn_btcq", "btc_attn"]], left_on="spike_date", right_index=True, how="left")

lo, hi = pd.Timestamp(PLACEBO_FROM), pd.Timestamp(PLACEBO_TO)
def make_blocked(pre, post):
    # exclude d in [s - pre, s_end + post]; s is any event day or identified spike (source day to spike day)
    bl = pd.Series(False, index=pd.date_range(lo - pd.Timedelta(days=400), hi + pd.Timedelta(days=400)))
    for _, e in events.iterrows():
        a = e.event_date
        b = (a + pd.offsets.MonthEnd(0)) if e.date_precision == "month" else a
        bl.loc[a - pd.Timedelta(days=pre): b + pd.Timedelta(days=post)] = True
    for _, r in audit.iterrows():
        bl.loc[r.source_date - pd.Timedelta(days=pre): r.spike_date + pd.Timedelta(days=post)] = True
    return bl

blocked     = make_blocked(PRE_EXCL, POST_EXCL)     # main: [s-30, s+14]
blocked_sym = make_blocked(EXCL_DAYS, EXCL_DAYS)    # robustness: [s-30, s+30]

qmax3 = att.attn_pc1.rolling(7, center=True, min_periods=1).max()      # maximum within +/-3 days
grid = pd.date_range(lo, hi, freq="W-TUE")
pool0 = [d for d in grid if not blocked.get(d, False)]
spiky = [d for d in pool0 if qmax3.get(d, 0) >= SPIKE_SD]
pool = [d for d in pool0 if d not in spiky]
pool_sym = [d for d in grid if not blocked_sym.get(d, False) and qmax3.get(d, 0) < SPIKE_SD]
print(f"{len(grid)} Tuesdays; after the main exclusion [s-{PRE_EXCL}, s+{POST_EXCL}] {len(pool0)}, minus {len(spiky)} quantum-attention spikes -> {len(pool)}")
print("  spike days removed:", [d.strftime("%Y-%m-%d") for d in spiky])
print("  main pool by year:", pd.Series([d.year for d in pool]).value_counts().sort_index().to_dict())
print(f"Robustness (symmetric +/-{EXCL_DAYS}): {len(pool_sym)} dates, by year:", pd.Series([d.year for d in pool_sym]).value_counts().sort_index().to_dict())

n_main = int((events.event_set == "main").sum())
def make_draws(p, name, seed):
    rng = np.random.default_rng(seed)
    out = pd.DataFrame([(name, i + 1, j + 1, d) for i in range(N_DRAWS)
                        for j, d in enumerate(sorted(rng.choice(p, size=n_main, replace=False)))],
                       columns=["pool", "draw_id", "slot", "fake_date"])
    out["fake_date"] = pd.to_datetime(out.fake_date)
    return out
draws = pd.concat([make_draws(pool, "main", SEED), make_draws(pool_sym, "sym30", SEED + 1)], ignore_index=True)

# non-quantum news days
thr = NQ_BTC_MIN
cand = att.loc[lo:hi].assign(qmax3=qmax3)
cand = cand[(cand.btc_attn >= thr) & (cand.qmax3 < 0.5) & (cand.index.dayofweek < 5)
            & ~cand.index.isin(blocked[blocked].index)]
picked = []
for d, r in cand.sort_values("btc_attn", ascending=False).iterrows():
    if all(abs((d - p).days) >= 30 for p in picked):
        picked.append(d)
    if len(picked) == n_main: break
print(f"Non-quantum news days: threshold btc_attn >= {thr:.2f} (fixed), {len(cand)} candidates, {len(picked)} selected")

placebo = pd.concat([
    pd.DataFrame({"kind": "ri_pool", "date": pool}),
    pd.DataFrame({"kind": "ri_pool_sym30", "date": pool_sym}),
    pd.DataFrame({"kind": "nonquantum_news", "date": sorted(picked)}),
], ignore_index=True)
placebo["date"] = pd.to_datetime(placebo.date)
placebo["snap_date"] = placebo.date - pd.Timedelta(days=1)
placebo = placebo.merge(att[["attn_pc1", "btc_attn"]], left_on="date", right_index=True, how="left")
display(placebo[placebo.kind == "nonquantum_news"].round(2))

upload("placebo_days", placebo)
upload("placebo_draws", draws)

# attention spikes for manual identification (not used as events automatically)
near = pd.Series(False, index=att.index)
for d0 in events.event_date:
    near.loc[d0 - pd.Timedelta(days=7): d0 + pd.Timedelta(days=7)] = True
sp = att.loc[(att.attn_pc1 >= 3) & ~near, ["attn_pc1", "attn_btcq", "btc_attn"]].sort_values("attn_pc1", ascending=False)
keep = []
for d0 in sp.index:
    if all(abs((d0 - k).days) >= 14 for k in keep): keep.append(d0)
spikes = sp.loc[sorted(keep)].reset_index().rename(columns={"index": "date"})
spikes["dow"] = spikes.date.dt.day_name()
# match with the audit table (within +/-3 days of the spike)
def _ident(d0):
    m = audit[(audit.spike_date - d0).abs() <= pd.Timedelta(days=3)]
    return m.label.iloc[0] if len(m) else ""
spikes["identified_as"] = spikes.date.map(_ident)
display(spikes.round(2))
print("Spikes not yet identified:", [d.strftime("%Y-%m-%d") for d in spikes.loc[spikes.identified_as == "", "date"]])
upload("attention_spikes", spikes)
display(audit.drop(columns="url").round(2))
upload("attention_spike_audit", audit)

## Checks

1. `events` matches the event dates of the 02 snapshots (asserted in step 1)
2. Attention to the main events: `p_btcq` (main), `p_pc1` (secondary); events with weak attention are kept
3. Correlation of the three attention measures (printed in step 4)
4. Pool size: about 30 dates or more for 1,000 draws of 6

In [ ]:
summary = {
    "events": events.groupby(["event_set", "date_status"]).size().rename("n").reset_index().to_dict(orient="records"),
    "main_overlaps": ov,
    "attention": {**PCA_INFO, "days": int(att.attn_pc1.notna().sum()),
                  "gtrends_in_btcq": "gtr_qbtc" in G_COLS},
    "event_attention": ev_att[["event", "peak_btcq", "p_btcq", "peak_pc1", "p_pc1", "attn_confounded", "peak_shared_with"]].round(3).to_dict(orient="records"),
    "attention_btcq_cols": BTCQ_COLS,
    "p_btcq<0.05 (primary)": ev_att.loc[ev_att.p_btcq < 0.05, "event"].tolist(),
    "p_pc1<0.05 (secondary)": ev_att.loc[ev_att.p_pc1 < 0.05, "event"].tolist(),
    "frozen_inputs": {t: table_rows(t) for t in ["wiki_frozen", "gtrends_frozen"]},
    "attention_spikes": [d.strftime("%Y-%m-%d") for d in spikes.date],
    "attention_spikes_unidentified": [d.strftime("%Y-%m-%d") for d in spikes.loc[spikes.identified_as == "", "date"]],
    "placebo": {"ri_pool": len(pool), "ri_pool_sym30": len(pool_sym),
                "ri_pool_by_year": pd.Series([d.year for d in pool]).value_counts().sort_index().to_dict(),
                "ri_pool_spikes_removed": [d.strftime("%Y-%m-%d") for d in spiky],
                "draws": N_DRAWS, "seed": SEED,
                "nonquantum_news": [d.strftime("%Y-%m-%d") for d in sorted(picked)]},
}
print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))

In [ ]:
for rec in JOBS[-5:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
for t in ["events", "event_windows", "gdelt_daily", "attention_daily", "event_attention", "placebo_days", "placebo_draws", "attention_spikes", "attention_spike_audit", "wiki_frozen", "gtrends_frozen"]:
    n = table_rows(t)
    print(f"{t:16s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(), "ledger": LEDGER,
          "tables": [table_info(t) for t in ["events", "event_windows", "gdelt_daily", "attention_daily",
                                              "event_attention", "placebo_days", "placebo_draws", "attention_spikes", "attention_spike_audit", "wiki_frozen", "gtrends_frozen"] if table_rows(t)]}
try:
    report["summary"] = summary
except NameError:
    pass
with open("build_report_03.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
try:
    from google.colab import files
    for fn in ["build_report_03.json", "fig_attention.png", "events.csv", "event_attention.csv", "placebo_days.csv",
               "attention_daily.csv", "pc1_decomposition.csv", "attention_spike_audit.csv",
               "wiki_frozen.csv", "gtrends_frozen.csv"]:
        if os.path.exists(fn): files.download(fn)
except Exception:
    print("Report saved to build_report_03.json")